# KSE-30 Incremental Update Pipeline

This notebook merges the previous extraction, sheet filtering, and company-name cleaning workflow into one KSE-30-only pipeline.

It will:
1. Read the stop date from `csvs/kse30_daily_data.csv`.
2. Download daily PSX files from the next date up to today.
3. Keep only sheet `KSE-30` / `KSE 30` (or close variants).
4. Standardize schema and symbol/company mapping.
5. Append deduplicated rows back into `csvs/kse30_daily_data.csv`.

In [ ]:
from __future__ import annotations

import os
import re
import time
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
# Paths
ROOT = Path(__file__).parent if '__file__' in globals() else Path('.').resolve()
CSV_PATH = ROOT / 'csvs' / 'kse30_daily_data.csv'
DOWNLOAD_DIR = ROOT / 'data' / 'kse30_incremental_raw'
KSE30_ONLY_DIR = ROOT / 'data' / 'kse30_only_sheets'
BACKUP_DIR = ROOT / 'data' / 'csv_backups'

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
KSE30_ONLY_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Matching pattern for target sheet
KSE30_PATTERN = re.compile(r'kse[\s\-_/]*30', re.IGNORECASE)

In [ ]:
def read_excel_any_engine(file_path: Path):
    """Try common engines so both .xls and .xlsx can be opened."""
    engines = ['openpyxl', 'xlrd', 'odf']
    for engine in engines:
        try:
            return pd.ExcelFile(file_path, engine=engine)
        except Exception:
            continue
    try:
        html_tables = pd.read_html(file_path)
        if html_tables:
            return {'HTML': html_tables[0]}
    except Exception:
        pass
    raise ValueError(f'Unable to open workbook: {file_path}')


def normalize_date_col(df: pd.DataFrame, fallback_date: date | None = None) -> pd.DataFrame:
    out = df.copy()

    date_col = None
    for candidate in ['DATE', 'Date', 'date']:
        if candidate in out.columns:
            date_col = candidate
            break

    if date_col is None:
        if fallback_date is None:
            raise ValueError('No date column found and no fallback date provided.')
        out['Date'] = fallback_date
    else:
        out['Date'] = pd.to_datetime(out[date_col], errors='coerce').dt.date
        if out['Date'].isna().all():
            if fallback_date is None:
                raise ValueError('Date column exists but could not be parsed.')
            out['Date'] = fallback_date

    return out


def canonical_company_map(df: pd.DataFrame) -> dict:
    data = df.copy()
    data['SYMBOL'] = data['SYMBOL'].astype(str).str.strip()
    data['COMPANY'] = data['COMPANY'].astype(str).str.strip()

    mapping = {}
    for sym, grp in data.groupby('SYMBOL', dropna=False):
        vals = grp['COMPANY'].dropna().astype(str).str.strip()
        vals = vals[vals != '']
        if len(vals) == 0:
            mapping[sym] = ''
            continue
        mode_vals = vals.mode()
        mapping[sym] = mode_vals.iloc[0] if len(mode_vals) > 0 else vals.iloc[0]
    return mapping


def coerce_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace(',', '', regex=False).str.replace('%', '', regex=False).str.strip()
    return pd.to_numeric(s, errors='coerce')


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Flatten potential unnamed columns
    out.columns = [str(c).strip() for c in out.columns]

    rename_map = {
        'Symbol': 'SYMBOL',
        'SYMBOL': 'SYMBOL',
        'Company': 'COMPANY',
        'COMPANY': 'COMPANY',
        'Price': 'PRICE',
        'PRICE': 'PRICE',
        'Idx Wt %': 'IDX WT %',
        'IDX WT %': 'IDX WT %',
        'FF Based Shares': 'FF BASED SHARES',
        'FF BASED SHARES': 'FF BASED SHARES',
        'FF Based Mcap': 'FF BASED MCAP',
        'FF BASED MCAP': 'FF BASED MCAP',
        'Ord Shares': 'ORD SHARES',
        'ORD SHARES': 'ORD SHARES',
        'Ord Shares Mcap': 'ORD SHARES MCAP',
        'ORD SHARES MCAP': 'ORD SHARES MCAP',
        'Volume': 'VOLUME',
        'VOLUME': 'VOLUME',
        'ISIN': 'ISIN'
    }

    out = out.rename(columns={c: rename_map.get(c, c) for c in out.columns})

    # Keep only relevant known columns if present
    preferred = [
        'Date', 'ISIN', 'SYMBOL', 'COMPANY', 'PRICE', 'IDX WT %',
        'FF BASED SHARES', 'FF BASED MCAP', 'ORD SHARES', 'ORD SHARES MCAP', 'VOLUME'
    ]
    keep = [c for c in preferred if c in out.columns]
    if keep:
        out = out[keep]

    return out

In [ ]:
def get_last_date(csv_path: Path) -> date:
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV not found: {csv_path}')

    df = pd.read_csv(csv_path)
    if 'Date' not in df.columns:
        raise ValueError("Expected 'Date' column in kse30_daily_data.csv")

    dates = pd.to_datetime(df['Date'], errors='coerce').dropna()
    if dates.empty:
        raise ValueError('No parseable dates in existing CSV.')

    return dates.max().date()


last_date = get_last_date(CSV_PATH)
start_date = last_date + timedelta(days=1)
end_date = date.today()

print('Existing CSV stop date:', last_date)
print('Download range:', start_date, 'to', end_date)

if start_date > end_date:
    print('CSV is already up-to-date; nothing to download.')

In [ ]:
def download_psx_files(start_date: date, end_date: date, out_dir: Path) -> list[Path]:
    if start_date > end_date:
        return []

    driver = webdriver.Chrome()
    downloaded = []

    try:
        url = 'https://dps.psx.com.pk/downloads'
        driver.get(url)

        current = start_date
        while current <= end_date:
            # Skip weekends
            if current.weekday() in (5, 6):
                current += timedelta(days=1)
                continue

            custom_date = current.strftime('%Y-%m-%d')
            print('Checking date', custom_date)

            try:
                date_field = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'input#downloadsDatePicker'))
                )
                date_field.clear()
                date_field.send_keys(custom_date)

                submit_button = driver.find_element(By.CSS_SELECTOR, 'button#downloadsSearchBtn')
                submit_button.click()
                time.sleep(2.5)

                link = driver.find_element(By.CSS_SELECTOR, 'div.downloads > div:nth-last-child(2) a.xls').get_attribute('href')
                if not link:
                    current += timedelta(days=1)
                    continue

                full_url = link if link.startswith('http') else f'https://dps.psx.com.pk{link}'
                file_name = os.path.basename(link)
                if not file_name.lower().endswith(('.xls', '.xlsx')):
                    file_name = f'{custom_date}.xls'

                local_path = out_dir / file_name
                response = requests.get(full_url, allow_redirects=True, timeout=30)
                response.raise_for_status()
                local_path.write_bytes(response.content)

                downloaded.append(local_path)
                print('Downloaded:', local_path.name)

            except NoSuchElementException:
                print('No downloadable file for', custom_date)
            except Exception as ex:
                print(f'Failed for {custom_date}: {ex}')

            current += timedelta(days=1)

    finally:
        driver.quit()

    return downloaded


downloaded_files = download_psx_files(start_date, end_date, DOWNLOAD_DIR)
print('Total downloaded files:', len(downloaded_files))

In [ ]:
def extract_kse30_sheet(input_path: Path, out_dir: Path) -> Path | None:
    try:
        xls = read_excel_any_engine(input_path)

        if isinstance(xls, dict):
            # Fallback HTML case, save as-is because there are no named sheets
            out_file = out_dir / (input_path.stem + '_kse30.xlsx')
            with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
                xls['HTML'].to_excel(writer, index=False, sheet_name='KSE-30')
            return out_file

        sheet_match = None
        for s in xls.sheet_names:
            norm = str(s).strip().replace(' ', ' ')
            if KSE30_PATTERN.search(norm):
                sheet_match = s
                break

        if sheet_match is None:
            print('No KSE-30 sheet in', input_path.name)
            return None

        df = pd.read_excel(xls, sheet_name=sheet_match)
        out_file = out_dir / (input_path.stem + '_kse30.xlsx')
        with pd.ExcelWriter(out_file, engine='openpyxl') as writer:
            df.to_excel(writer, index=False, sheet_name='KSE-30')

        return out_file

    except Exception as ex:
        print('Sheet extraction failed for', input_path.name, ':', ex)
        return None


kse30_sheet_files = []
for f in sorted(DOWNLOAD_DIR.glob('*')):
    if f.suffix.lower() not in ('.xls', '.xlsx'):
        continue
    out = extract_kse30_sheet(f, KSE30_ONLY_DIR)
    if out is not None:
        kse30_sheet_files.append(out)

print('KSE-30 extracted files:', len(kse30_sheet_files))

In [ ]:
def infer_date_from_filename(path: Path) -> date | None:
    # Looks for YYYY-MM-DD in file name
    m = re.search(r'(\d{4}-\d{2}-\d{2})', path.stem)
    if not m:
        return None
    try:
        return datetime.strptime(m.group(1), '%Y-%m-%d').date()
    except Exception:
        return None


def read_kse30_daily_rows(file_path: Path) -> pd.DataFrame:
    xls = read_excel_any_engine(file_path)

    if isinstance(xls, dict):
        df = xls['HTML'].copy()
    else:
        df = pd.read_excel(xls, sheet_name=0)

    df = standardize_columns(df)
    fallback = infer_date_from_filename(file_path)
    df = normalize_date_col(df, fallback_date=fallback)

    # Keep only rows that look like constituents (must have SYMBOL)
    if 'SYMBOL' in df.columns:
        df['SYMBOL'] = df['SYMBOL'].astype(str).str.strip()
        df = df[df['SYMBOL'].notna() & (df['SYMBOL'] != '')]

    if 'COMPANY' in df.columns:
        df['COMPANY'] = df['COMPANY'].astype(str).str.strip()

    for col in ['PRICE', 'IDX WT %', 'FF BASED SHARES', 'FF BASED MCAP', 'ORD SHARES', 'ORD SHARES MCAP', 'VOLUME']:
        if col in df.columns:
            df[col] = coerce_numeric(df[col])

    return df


new_parts = []
for fp in sorted(KSE30_ONLY_DIR.glob('*.xlsx')):
    try:
        part = read_kse30_daily_rows(fp)
        if not part.empty:
            new_parts.append(part)
    except Exception as ex:
        print('Failed parsing', fp.name, ':', ex)

new_df = pd.concat(new_parts, ignore_index=True) if new_parts else pd.DataFrame()
print('New parsed rows:', len(new_df))
display(new_df.head()) if not new_df.empty else None

In [ ]:
existing_df = pd.read_csv(CSV_PATH)

# Align existing schema
existing_df = standardize_columns(existing_df)
existing_df = normalize_date_col(existing_df)

if new_df.empty:
    final_df = existing_df.copy()
    print('No new rows detected; existing CSV kept unchanged.')
else:
    # Add missing columns on either side
    all_cols = sorted(set(existing_df.columns).union(set(new_df.columns)))
    existing_aligned = existing_df.reindex(columns=all_cols)
    new_aligned = new_df.reindex(columns=all_cols)

    combined = pd.concat([existing_aligned, new_aligned], ignore_index=True)

    # Canonical company name mapping by symbol across full history
    if 'SYMBOL' in combined.columns and 'COMPANY' in combined.columns:
        cmap = canonical_company_map(combined[['SYMBOL', 'COMPANY']].copy())
        combined['SYMBOL'] = combined['SYMBOL'].astype(str).str.strip()
        combined['COMPANY'] = combined['SYMBOL'].map(cmap).fillna('')

    # Deduplicate rows by date + symbol (keep latest occurrence)
    if 'Date' in combined.columns and 'SYMBOL' in combined.columns:
        combined = combined.drop_duplicates(subset=['Date', 'SYMBOL'], keep='last')

    # Sort by date then symbol
    combined['Date'] = pd.to_datetime(combined['Date'], errors='coerce').dt.date
    combined = combined.sort_values(['Date', 'SYMBOL'], kind='stable').reset_index(drop=True)

    final_df = combined

print('Existing rows:', len(existing_df))
print('Final rows   :', len(final_df))
print('Final max date:', pd.to_datetime(final_df['Date'], errors='coerce').max())
display(final_df.tail(10))

In [ ]:
# Backup + write updated CSV
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_path = BACKUP_DIR / f'kse30_daily_data_backup_{timestamp}.csv'

if CSV_PATH.exists():
    CSV_PATH.replace(backup_path)
    print('Backup created:', backup_path)

# Put Date first and keep common order when possible
preferred_order = [
    'Date', 'ISIN', 'SYMBOL', 'COMPANY', 'PRICE', 'IDX WT %',
    'FF BASED SHARES', 'FF BASED MCAP', 'ORD SHARES', 'ORD SHARES MCAP', 'VOLUME'
]
ordered_cols = [c for c in preferred_order if c in final_df.columns] + [c for c in final_df.columns if c not in preferred_order]

final_df = final_df[ordered_cols]
final_df.to_csv(CSV_PATH, index=False)
print('Updated CSV written:', CSV_PATH)

## Notes

- This notebook is KSE-30-only and intentionally does not use `3-separate-data.py`.
- If Selenium cannot find ChromeDriver automatically, install a compatible ChromeDriver and keep it in PATH.
- If PSX page structure changes, update CSS selectors in `download_psx_files`.
- The notebook creates a timestamped backup before overwriting `csvs/kse30_daily_data.csv`.